In [0]:
"""
id: source_a1b2c3d5
template: source
templateVersion: 2.0.0
name: gold_employees_current_3
parentId: group_c3d4e5f6
position:
  x: 1530
  y: 233
codegenHash: f9bd80fc93b51b62
config:
  table_source:
    tableName: alp_demo_catalog.acme_hr.gold_employees_current
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "alp_demo_catalog.acme_hr.gold_employees_current"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_a1b2c3d5.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_a1b2c3d5" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_a1b2c3d5.data"])
if "source_a1b2c3d5" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_a1b2c3d5", "counts": {"data": int(ctx["source_a1b2c3d5.data"].count())}}}, raw=True)

In [0]:
"""
id: source_d0e1f2a3
template: source
templateVersion: 2.0.0
name: gold_employees_current_2
parentId: group_b2c3d4e5
position:
  x: 30
  y: 233
description:
  text: Read all data from the table of current employees.
  hash: a4cb058d
codegenHash: f9bd80fc93b51b62
config:
  table_source:
    tableName: alp_demo_catalog.acme_hr.gold_employees_current
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "alp_demo_catalog.acme_hr.gold_employees_current"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_d0e1f2a3.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_d0e1f2a3" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_d0e1f2a3.data"])
if "source_d0e1f2a3" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_d0e1f2a3", "counts": {"data": int(ctx["source_d0e1f2a3.data"].count())}}}, raw=True)

In [0]:
"""
id: source_e1f2a3b4
template: source
templateVersion: 2.0.0
name: gold_employees_current
parentId: group_a1b2c3d4
position:
  x: 30
  y: 78
description:
  text: Read all data from the specified employee table.
  hash: a893d599
codegenHash: f9bd80fc93b51b62
config:
  table_source:
    tableName: alp_demo_catalog.acme_hr.gold_employees_current
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "alp_demo_catalog.acme_hr.gold_employees_current"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_e1f2a3b4.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_e1f2a3b4" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_e1f2a3b4.data"])
if "source_e1f2a3b4" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_e1f2a3b4", "counts": {"data": int(ctx["source_e1f2a3b4.data"].count())}}}, raw=True)

In [0]:
"""
id: source_f6a7b8c9
template: source
templateVersion: 2.0.0
name: silver_absences
parentId: group_b2c3d4e5
position:
  x: 30
  y: 78
description:
  text: Read all data from the specified table.
  hash: a1d70e8d
codegenHash: f9bd80fc93b51b62
config:
  table_source:
    tableName: alp_demo_catalog.acme_hr.silver_absences
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "alp_demo_catalog.acme_hr.silver_absences"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_f6a7b8c9.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_f6a7b8c9" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_f6a7b8c9.data"])
if "source_f6a7b8c9" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_f6a7b8c9", "counts": {"data": int(ctx["source_f6a7b8c9.data"].count())}}}, raw=True)

In [0]:
"""
id: source_g7h8i9j0
template: source
templateVersion: 2.0.0
name: silver_absences_2
parentId: group_c3d4e5f6
position:
  x: 30
  y: 78
codegenHash: f9bd80fc93b51b62
config:
  table_source:
    tableName: alp_demo_catalog.acme_hr.silver_absences
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "alp_demo_catalog.acme_hr.silver_absences"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_g7h8i9j0.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_g7h8i9j0" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_g7h8i9j0.data"])
if "source_g7h8i9j0" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_g7h8i9j0", "counts": {"data": int(ctx["source_g7h8i9j0.data"].count())}}}, raw=True)

In [0]:
"""
id: filter_c5d6e7f8
template: filter
templateVersion: 2.0.0
name: Active employees
parentId: group_a1b2c3d4
position:
  x: 330
  y: 78
description:
  text: Split data into two groups based on a condition.
  hash: b8807b50
codegenHash: d8c91e247e7ed9ff
config:
  condition: status = 'Active'
input:
  - node: source_e1f2a3b4
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "status = 'Active'"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_e1f2a3b4.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["filter_c5d6e7f8.filtered_data"] = out["filtered_data"]
ctx["filter_c5d6e7f8.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False) or "filter_c5d6e7f8" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["filter_c5d6e7f8.filtered_data"])
    display(ctx["filter_c5d6e7f8.excluded_data"])
if "filter_c5d6e7f8" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "filter_c5d6e7f8", "counts": {"filtered_data": int(ctx["filter_c5d6e7f8.filtered_data"].count()), "excluded_data": int(ctx["filter_c5d6e7f8.excluded_data"].count())}}}, raw=True)

In [0]:
"""
id: join_b4c5d6e7
template: join
templateVersion: 3.0.0
name: Absences with employees
parentId: group_b2c3d4e5
position:
  x: 330
  y: 155.5
description:
  text: Combine two tables by matching employee IDs and show matched and unmatched records.
  hash: 1a72ebe7
codegenHash: ea5821d113ebbb3b
config:
  join_type: inner
  match_case: false
  multi_table: false
  conditions:
    - left_input: 0
      right_input: 1
      join_keys:
        - left: employee_id
          right: employee_id
      join_conditions: ""
  columns:
    - edits: []
      ordered: []
    - edits: []
      ordered: []
input:
  - node: source_f6a7b8c9
    input_port: left
    output_port: data
  - node: source_d0e1f2a3
    input_port: right
    output_port: data
"""

# generated from the system
from typing import Any, Dict, List, Optional
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(alias: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + alias + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(frames: List[Any]):
    out = []
    seen: Dict[str, int] = {}
    n = len(frames)
    for idx, (alias, df, cfg) in enumerate(frames):
        edits = _edits_by_column(cfg)
        for name in _ordered_names(list(df.columns), cfg):
            edit = edits.get(name)
            if edit is not None and not _is_checked(edit):
                continue
            col = _qualified_col(alias, name)
            user_alias = edit.get("alias") if edit else None
            if user_alias:
                out.append(col.alias(user_alias))
                seen[user_alias.lower()] = seen.get(user_alias.lower(), 0) + 1
                continue
            key = name.lower()
            if key not in seen:
                seen[key] = 1
                out.append(col.alias(name))
                continue
            new_name = ("right_" + name) if (n == 2 and idx == 1) else (name + "_" + str(idx + 1))
            while new_name.lower() in seen:
                new_name = new_name + "_dup"
            seen[new_name.lower()] = 1
            out.append(col.alias(new_name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = []
    for port in ("left", "right"):
        df = inputs.get(port)
        if df is not None:
            dfs.append(df)
    dfs.extend(inputs.get("data") or [])
    if len(dfs) < 2:
        raise ValueError(
            f"Join requires at least 2 inputs, got {len(dfs)}. "
            "Wire 2 or more upstream operators into this Join."
        )

    n = len(dfs)
    join_type = config.get("join_type") or "split_join"
    match_case = bool(config.get("match_case", False))
    conditions: List[Dict[str, Any]] = config.get("conditions") or []
    columns: List[Dict[str, Any]] = config.get("columns") or []

    aliased = [df.alias("t" + str(i)) for i, df in enumerate(dfs)]
    types = [{f.name.lower(): f.dataType for f in df.schema} for df in aliased]

    def key_col(idx: int, name: str):
        col = _qualified_col("t" + str(idx), name)
        if not match_case and isinstance(types[idx].get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    def edge_predicate(i: int, j: int, edge: Dict[str, Any]) -> Optional[Any]:
        predicates = []
        for key in edge.get("join_keys") or []:
            predicates.append(key_col(i, key["left"]) == key_col(j, key["right"]))
        condition = edge.get("join_conditions") or ""
        if condition:
            predicates.append(F.expr(condition))
        join_expr = None
        for predicate in predicates:
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    edges: List[Any] = [
        (int(edge.get("left_input", 0)), int(edge.get("right_input", 0)), edge)
        for edge in conditions
    ]

    predicates_by_step: Dict[int, List[Any]] = {}
    for i, j, edge in edges:
        predicate = edge_predicate(i, j, edge)
        if predicate is not None:
            predicates_by_step.setdefault(max(i, j), []).append(predicate)

    def predicate_at(step: int) -> Optional[Any]:
        join_expr = None
        for predicate in predicates_by_step.get(step, []):
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    def cfg(i: int) -> Dict[str, Any]:
        return columns[i] if i < len(columns) else {}

    is_two = n == 2
    is_split = is_two and join_type == "split_join"

    if is_split:
        join_expr = predicate_at(1)
        left_df, right_df = aliased[0], aliased[1]
        if join_expr is None:
            matched = left_df.join(right_df, how="inner")
            left_unmatched = left_df.join(right_df, how="left_anti")
            right_unmatched = right_df.join(left_df, how="left_anti")
        else:
            matched = left_df.join(right_df, join_expr, how="inner")
            left_unmatched = left_df.join(right_df, join_expr, how="left_anti")
            right_unmatched = right_df.join(left_df, join_expr, how="left_anti")
        projection = _projection([("t0", left_df, cfg(0)), ("t1", right_df, cfg(1))])
        if projection:
            matched = matched.select(*projection)
        return {
            "joined_data": matched,
            "left_unmatched": left_unmatched,
            "right_unmatched": right_unmatched,
        }

    how = join_type if is_two else "inner"
    if is_two:
        acc = aliased[0]
        predicate = predicate_at(1)
        if predicate is None:
            acc = acc.join(aliased[1], how=how)
        else:
            acc = acc.join(aliased[1], predicate, how=how)
    else:
        adjacency: Dict[int, List[int]] = {}
        for idx, (i, j, _edge) in enumerate(edges):
            adjacency.setdefault(i, []).append(idx)
            adjacency.setdefault(j, []).append(idx)
        visited = [False] * n
        visited[0] = True
        acc = aliased[0]
        used_edges = set()
        queue: List[int] = [0]
        while queue:
            cur = queue.pop(0)
            for idx in adjacency.get(cur, []):
                i, j, edge = edges[idx]
                nxt = j if i == cur else i
                if visited[nxt]:
                    continue
                visited[nxt] = True
                used_edges.add(idx)
                predicate = edge_predicate(i, j, edge)
                if predicate is None:
                    acc = acc.join(aliased[nxt], how=how)
                else:
                    acc = acc.join(aliased[nxt], predicate, how=how)
                queue.append(nxt)
        for idx, (i, j, edge) in enumerate(edges):
            if idx in used_edges:
                continue
            predicate = edge_predicate(i, j, edge)
            if predicate is not None:
                acc = acc.where(predicate)

    projection = _projection([("t" + str(i), aliased[i], cfg(i)) for i in range(n)])
    if projection:
        acc = acc.select(*projection)

    return {
        "joined_data": acc,
        "left_unmatched": spark.createDataFrame([], aliased[0].schema),
        "right_unmatched": spark.createDataFrame([], aliased[n - 1].schema),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "inner",
    "match_case": False,
    "multi_table": False,
    "conditions": [
        {
            "left_input": 0,
            "right_input": 1,
            "join_keys": [
                {
                    "left": "employee_id",
                    "right": "employee_id"
                }
            ],
            "join_conditions": ""
        }
    ],
    "columns": [
        {
            "edits": [],
            "ordered": []
        },
        {
            "edits": [],
            "ordered": []
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "left": ctx["source_f6a7b8c9.data"],
    "right": ctx["source_d0e1f2a3.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["join_b4c5d6e7.joined_data"] = out["joined_data"]
ctx["join_b4c5d6e7.left_unmatched"] = out["left_unmatched"]
ctx["join_b4c5d6e7.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", False) or "join_b4c5d6e7" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["join_b4c5d6e7.joined_data"])
    display(ctx["join_b4c5d6e7.left_unmatched"])
    display(ctx["join_b4c5d6e7.right_unmatched"])
if "join_b4c5d6e7" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "join_b4c5d6e7", "counts": {"joined_data": int(ctx["join_b4c5d6e7.joined_data"].count()), "left_unmatched": int(ctx["join_b4c5d6e7.left_unmatched"].count()), "right_unmatched": int(ctx["join_b4c5d6e7.right_unmatched"].count())}}}, raw=True)

In [0]:
"""
id: filter_k1l2m3n4
template: filter
templateVersion: 2.0.0
name: Non-null comments
parentId: group_c3d4e5f6
position:
  x: 330
  y: 121.75
codegenHash: d8c91e247e7ed9ff
config:
  condition: comment IS NOT NULL
input:
  - node: source_g7h8i9j0
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "comment IS NOT NULL"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_g7h8i9j0.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["filter_k1l2m3n4.filtered_data"] = out["filtered_data"]
ctx["filter_k1l2m3n4.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False) or "filter_k1l2m3n4" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["filter_k1l2m3n4.filtered_data"])
    display(ctx["filter_k1l2m3n4.excluded_data"])
if "filter_k1l2m3n4" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "filter_k1l2m3n4", "counts": {"filtered_data": int(ctx["filter_k1l2m3n4.filtered_data"].count()), "excluded_data": int(ctx["filter_k1l2m3n4.excluded_data"].count())}}}, raw=True)

In [0]:
"""
id: sql_a8b9c0d1
template: sql
templateVersion: 1.0.0
name: Headcount metrics
parentId: group_a1b2c3d4
position:
  x: 630
  y: 78
description:
  text: Summarize employee data by department and region, including headcount, average age, average seniority, count of permanent contracts, and female representation.
  hash: ea09b992
codegenHash: 93d0fe419523af76
config:
  query: |
    SELECT
      department_id,
      department_name,
      business_unit,
      region,
      COUNT(*) AS headcount,
      ROUND(AVG(age), 1) AS avg_age,
      ROUND(AVG(CAST(seniority_years AS DOUBLE)), 1) AS avg_seniority_years,
      SUM(CASE WHEN contract_type = 'CDI' THEN 1 ELSE 0 END) AS cdi_count,
      SUM(CASE WHEN gender = 'F' THEN 1 ELSE 0 END) AS female_count,
      ROUND(
        SUM(CASE WHEN gender = 'F' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1
      ) AS female_pct
    FROM `Active_employees`
    GROUP BY
      department_id,
      department_name,
      business_unit,
      region
input:
  - node: filter_c5d6e7f8
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT\n  department_id,\n  department_name,\n  business_unit,\n  region,\n  COUNT(*) AS headcount,\n  ROUND(AVG(age), 1) AS avg_age,\n  ROUND(AVG(CAST(seniority_years AS DOUBLE)), 1) AS avg_seniority_years,\n  SUM(CASE WHEN contract_type = 'CDI' THEN 1 ELSE 0 END) AS cdi_count,\n  SUM(CASE WHEN gender = 'F' THEN 1 ELSE 0 END) AS female_count,\n  ROUND(\n    SUM(CASE WHEN gender = 'F' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),\n    1\n  ) AS female_pct\nFROM `Active_employees`\nGROUP BY\n  department_id,\n  department_name,\n  business_unit,\n  region\n"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": [
        ctx["filter_c5d6e7f8.filtered_data"]
    ],
    "data__sources": [
        {
            "node": "filter_c5d6e7f8",
            "output_port": "filtered_data",
            "name": "Active employees",
            "df_name": "Active_employees"
        }
    ]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["sql_a8b9c0d1.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "sql_a8b9c0d1" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["sql_a8b9c0d1.result"])
if "sql_a8b9c0d1" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "sql_a8b9c0d1", "counts": {"result": int(ctx["sql_a8b9c0d1.result"].count())}}}, raw=True)

In [0]:
"""
id: sql_e8f9a0b1
template: aggregate
templateVersion: 2.0.0
name: Absenteeism metrics
parentId: group_b2c3d4e5
position:
  x: 630
  y: 155.5
description:
  text: Group data by department, business unit, month, and absence type, then calculate distinct absent employees, total absence events, total absence days, and average days per absence.
  hash: 87f9a0aa
codegenHash: 7e9149ee54d999b7
config:
  group_bys:
    - expr: department_id
      type: expr
    - expr: department_name
      type: expr
    - expr: business_unit
      type: expr
    - expr: absence_month
      type: expr
    - expr: absence_type
      type: expr
  aggregations:
    - columnExpr:
        expr: employee_id
        type: expr
      fn: COUNT_DISTINCT
      alias: employees_absent
    - columnExpr:
        expr: employee_id
        type: expr
      fn: COUNT
      alias: absence_events
    - columnExpr:
        expr: days
        type: expr
      fn: SUM
      alias: total_absence_days
    - columnExpr:
        expr: days
        type: expr
      fn: AVG
      alias: avg_days_per_absence
input:
  - node: join_b4c5d6e7
    input_port: data
    output_port: joined_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = agg_fn(arg)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            concat_arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = F.concat_ws(sep, F.collect_list(concat_arg))
        elif fn == "COUNT_DISTINCT":
            arg = F.expr(raw_expr) if col_expr.get("type") != "column" else raw_expr
            col = F.count_distinct(arg)
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "department_id",
            "type": "expr"
        },
        {
            "expr": "department_name",
            "type": "expr"
        },
        {
            "expr": "business_unit",
            "type": "expr"
        },
        {
            "expr": "absence_month",
            "type": "expr"
        },
        {
            "expr": "absence_type",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "employee_id",
                "type": "expr"
            },
            "fn": "COUNT_DISTINCT",
            "alias": "employees_absent",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "employee_id",
                "type": "expr"
            },
            "fn": "COUNT",
            "alias": "absence_events",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "days",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "total_absence_days",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "days",
                "type": "expr"
            },
            "fn": "AVG",
            "alias": "avg_days_per_absence",
            "withAsKeyword": None
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["join_b4c5d6e7.joined_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["sql_e8f9a0b1.aggregated_data"] = out["aggregated_data"]
if globals().get("ld_display_outputs", False) or "sql_e8f9a0b1" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["sql_e8f9a0b1.aggregated_data"])
if "sql_e8f9a0b1" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "sql_e8f9a0b1", "counts": {"aggregated_data": int(ctx["sql_e8f9a0b1.aggregated_data"].count())}}}, raw=True)

In [0]:
"""
id: unique_o5p6q7r8
template: unique
templateVersion: 1.0.0
name: Unique comments
parentId: group_c3d4e5f6
position:
  x: 630
  y: 121.75
previewCodeHash: 72fb75b700428e88
codegenHash: f37e4cbd06314ad6
config:
  unique_by_all_columns: false
  columns:
    - comment
  sort_expressions: []
input:
  - node: filter_k1l2m3n4
    input_port: data
    output_port: filtered_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F
from pyspark.sql import Window

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    if df is None:
        raise ValueError("Unique operator requires input 'data'")
    unique_by_all_columns = config.get("unique_by_all_columns", True)
    columns = config.get("columns", [])
    if not isinstance(columns, list):
        columns = []
    else:
        columns = [c for c in columns if isinstance(c, str)]
    sort_expressions = config.get("sort_expressions", [])

    if unique_by_all_columns:
        columns = df.columns
    elif not columns:
        return {"unique_data": df}

    order_idx = "__lb_orig_order__"
    while order_idx in df.columns:
        order_idx = order_idx + "_"
    df = df.withColumn(order_idx, F.monotonically_increasing_id())

    if unique_by_all_columns or not sort_expressions:
        deduped = df.dropDuplicates(columns)
        return {"unique_data": deduped.orderBy(order_idx).drop(order_idx)}

    order_cols = []
    for sort_def in sort_expressions:
        if not isinstance(sort_def, dict):
            continue
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        if not raw_expr:
            continue
        direction = sort_def.get("sortBy", "ASC")
        col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        if direction == "DESC":
            col = col.desc_nulls_last()
        elif direction == "ASC":
            col = col.asc_nulls_last()
        order_cols.append(col)
    
    if not order_cols:
        deduped = df.dropDuplicates(columns)
        return {"unique_data": deduped.orderBy(order_idx).drop(order_idx)}

    rn = "__lb_row_number__"
    while rn in df.columns:
        rn = rn + "_"

    window = Window.partitionBy(*[F.col(c) for c in columns]).orderBy(*order_cols, F.col(order_idx))
    tagged = df.withColumn(rn, F.row_number().over(window))
    unique = tagged.filter(F.col(rn) == 1).orderBy(order_idx).drop(rn, order_idx)
    return {"unique_data": unique}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "unique_by_all_columns": False,
    "columns": [
        "comment"
    ],
    "sort_expressions": []
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["filter_k1l2m3n4.filtered_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["unique_o5p6q7r8.unique_data"] = out["unique_data"]
if globals().get("ld_display_outputs", False) or "unique_o5p6q7r8" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["unique_o5p6q7r8.unique_data"])
if "unique_o5p6q7r8" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "unique_o5p6q7r8", "counts": {"unique_data": int(ctx["unique_o5p6q7r8.unique_data"].count())}}}, raw=True)

In [0]:
"""
id: output_d2e3f4a5
template: output
templateVersion: 4.0.0
name: gold_headcount_by_department
parentId: group_a1b2c3d4
position:
  x: 930
  y: 78
description:
  text: Create or refresh a materialized view named gold_headcount_by_department in alp_demo_catalog.acme_hr using all data from the input.
  hash: f1860d77
codegenHash: e556653ac6e0f4e0
config:
  output_type: materialized_view
  catalog: alp_demo_catalog
  schema: acme_hr
  view_name: gold_headcount_by_department
  cluster_by_mode: auto
input:
  - node: sql_a8b9c0d1
    input_port: data
    output_port: result
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_FAST_CSV_CELLS = 50_000_000

MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES = 2 * 1024 * 1024 * 1024

_FAST_CSV_BYTES_PER_CELL_OVERHEAD = 16

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _pandas_line_terminator_parameter(pd) -> str:
    import inspect

    return (
        "lineterminator"
        if "lineterminator" in inspect.signature(pd.DataFrame.to_csv).parameters
        else "line_terminator"
    )

def _write_csv_overwrite(df, path: str, include_header: bool = True) -> bool:
    import os
    import shutil
    import tempfile

    from pyspark.sql import functions as F
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return False

    supported_type_names = {
        "boolean",
        "byte",
        "char",
        "date",
        "decimal",
        "double",
        "float",
        "integer",
        "long",
        "short",
        "string",
        "timestamp",
        "timestamp_ntz",
        "varchar",
        "void",
    }
    normalized_names = [name.casefold() for name in df.columns]
    if (
        not df.columns
        or len(set(normalized_names)) != len(normalized_names)
        or any(field.dataType.typeName() not in supported_type_names for field in df.schema.fields)
    ):
        return False

    spark_output_directory = os.path.isdir(path) and os.path.isfile(
        os.path.join(path, "_SUCCESS")
    )
    if os.path.isdir(path) and not spark_output_directory:
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )

    from databricks.sdk import WorkspaceClient

    client = WorkspaceClient()

    expressions = []
    output_column_names = []
    nan_markers = []
    for index, field in enumerate(df.schema.fields):
        escaped_name = field.name.replace("`", "``")
        column = F.col(f"`{escaped_name}`")
        output_column_name = f"__lb_csv_column_{index}"
        output_column_names.append(field.name)
        type_name = field.dataType.typeName()
        if type_name in {"char", "string", "varchar"}:
            first_character = F.substring(column, 1, 1)
            column = F.when(
                first_character.isin(*_FORMULA_TRIGGERS),
                F.concat(F.lit("'"), column),
            ).otherwise(column)
        elif type_name in {"byte", "integer", "long", "short"}:
            column = column.cast("string")
        elif type_name in {"double", "float"}:
            marker = f"__lb_csv_nan_{index}"
            nan_markers.append((output_column_name, marker))
            expressions.append(F.isnan(column).alias(marker))
        elif type_name == "boolean":
            column = F.when(column, F.lit("True")).when(~column, F.lit("False"))
        elif type_name in {"date", "decimal"}:
            column = column.cast("string")
        elif type_name in {"timestamp", "timestamp_ntz"}:
            column = F.regexp_replace(
                F.date_format(column, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
                r"[.]000000$",
                "",
            )
        expressions.append(column.alias(output_column_name))

    fast_csv_row_cap = min(
        MAX_SINGLE_FILE_ROWS,
        max(1, MAX_FAST_CSV_CELLS // len(df.columns)),
    )
    prepared_df = df.select(*expressions).limit(fast_csv_row_cap + 1).persist()
    try:
        payload_size = F.lit(0)
        for name in prepared_df.columns:
            escaped_name = name.replace("`", "``")
            payload_size = payload_size + F.length(
                F.encode(
                    F.coalesce(F.col(f"`{escaped_name}`").cast("string"), F.lit("")),
                    "UTF-8",
                )
            )
        stats = prepared_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.sum(payload_size).alias("payload_bytes"),
        ).collect()[0]
        row_count = stats["row_count"]
        if row_count > fast_csv_row_cap:
            if fast_csv_row_cap < MAX_SINGLE_FILE_ROWS:
                raise ValueError(
                    f"Output: result too wide for safe single-file CSV export "
                    f"(over {MAX_FAST_CSV_CELLS:,} cells) — write to a table instead."
                )
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
        estimated_driver_bytes = (stats["payload_bytes"] or 0) + (
            row_count * len(prepared_df.columns) * _FAST_CSV_BYTES_PER_CELL_OVERHEAD
        )
        if estimated_driver_bytes > MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES:
            raise ValueError(
                f"Output: result too large for safe single-file CSV export "
                f"(estimated driver memory exceeds "
                f"{MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES // (1024 * 1024):,} MiB) — "
                f"write to a table instead."
            )
        pdf = prepared_df.toPandas()
    finally:
        prepared_df.unpersist()
    for column_name, marker in nan_markers:
        nan_mask = pdf[marker].fillna(False)
        finite_mask = ~pdf[column_name].isna()
        pdf[column_name] = pdf[column_name].astype(object)
        pdf.loc[finite_mask, column_name] = pdf.loc[finite_mask, column_name].map(
            lambda value: repr(float(value))
        )
        pdf.loc[nan_mask, column_name] = "nan"
        pdf.loc[~nan_mask & ~finite_mask, column_name] = None
        pdf.drop(columns=[marker], inplace=True)
    pdf.columns = output_column_names
    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        pdf.to_csv(
            local_tmp,
            index=False,
            header=include_header,
            **{_pandas_line_terminator_parameter(pd): "\r\n"},
        )
        if spark_output_directory:
            shutil.rmtree(path)
        with open(local_tmp, "rb") as f:
            client.files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)
    return True

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    if (
        file_type == "csv"
        and not split_by_column
        and write_mode == "overwrite"
        and config.get("__fast_csv_overwrite", False)
        and _write_csv_overwrite(df, path, include_header)
    ):
        return

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "materialized_view",
    "catalog": "alp_demo_catalog",
    "schema": "acme_hr",
    "view_name": "gold_headcount_by_department",
    "cluster_by_mode": "auto"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["sql_a8b9c0d1.result"]
}
if config["meta_state"]["is_disabled"]:
    try:
        _lb_disabled_persisted_result = spark.sql("SELECT * FROM alp_demo_catalog.acme_hr.gold_headcount_by_department").limit(0)
        _lb_disabled_persisted_result.schema
    except Exception as _lb_disabled_persisted_error:
        _lb_disabled_persisted_condition_getter = getattr(_lb_disabled_persisted_error, "getCondition", None)
        _lb_disabled_persisted_error_class = _lb_disabled_persisted_condition_getter() if callable(_lb_disabled_persisted_condition_getter) else None
        if not _lb_disabled_persisted_error_class:
            _lb_disabled_persisted_legacy_error_class_getter = getattr(_lb_disabled_persisted_error, "getErrorClass", None)
            _lb_disabled_persisted_error_class = _lb_disabled_persisted_legacy_error_class_getter() if callable(_lb_disabled_persisted_legacy_error_class_getter) else None
        if _lb_disabled_persisted_error_class not in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND"):
            raise
        _lb_disabled_persisted_input = inputs.get("data")
        _lb_disabled_persisted_result = _lb_disabled_persisted_input.limit(0) if hasattr(_lb_disabled_persisted_input, "limit") else _lb_disabled_persisted_input
    out = {"result": _lb_disabled_persisted_result}
else:
    out = run(config, inputs, spark)
ctx["output_d2e3f4a5.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_d2e3f4a5" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_d2e3f4a5.result"])
if "output_d2e3f4a5" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_d2e3f4a5", "counts": {"result": int(ctx["output_d2e3f4a5.result"].count())}}}, raw=True)

In [0]:
"""
id: prepare_b5c6d7e8
template: prepare
templateVersion: 1.0.0
name: Round avg days
parentId: group_b2c3d4e5
position:
  x: 930
  y: 155.5
description:
  text: Round avg_days_per_absence to 1 decimal place.
  hash: 63aee398
previewCodeHash: a2b9815bd44ababc
codegenHash: 0517a7ccfa379cb8
config:
  actions:
    - type: formula
      target: avg_days_per_absence
      expression: ROUND(avg_days_per_absence, 1)
input:
  - node: sql_e8f9a0b1
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "avg_days_per_absence",
            "expression": "ROUND(avg_days_per_absence, 1)"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["sql_e8f9a0b1.aggregated_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_b5c6d7e8.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_b5c6d7e8" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_b5c6d7e8.prepared_data"])
if "prepare_b5c6d7e8" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "prepare_b5c6d7e8", "counts": {"prepared_data": int(ctx["prepare_b5c6d7e8.prepared_data"].count())}}}, raw=True)

In [0]:
"""
id: ai_function_20
template: ai_function
templateVersion: 3.0.0
name: Classify Reason
parentId: group_c3d4e5f6
position:
  x: 930
  y: 121.75
description:
  text: Add a column classifying comments into reason categories while keeping all original data columns.
  hash: 3c7c58a5
codegenHash: 2cbe24e3809a0411
config:
  expressions:
    - ai_classify(comment, '["santé","famille","formation","lié au travail","personnel"]') `reason_category_json`
  keep_all_columns: true
input:
  - node: unique_o5p6q7r8
    input_port: data
    output_port: unique_data
"""

# generated from the system
from typing import Dict, Any, List
import hashlib

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    # Table-valued AI functions (ai_forecast, ...) live in the FROM
    # clause and have no SELECT-list shape, so we run them via
    # spark.sql and bind the upstream DataFrame as a session-scoped
    # temp view substituted in for the
    # __lakebuilder_ai_function_input__ placeholder identifier
    # (which is a real SQL identifier so the persisted statement
    # round-trips through the SQL parser when the cell reloads).
    #
    # spark.sql returns a lazy DataFrame: the view name is resolved
    # by Spark at action time (preview limit/collect), not when
    # spark.sql is called. We therefore deliberately do NOT drop
    # the temp view here — dropping it would leave the lazy plan
    # pointing at a missing relation and trigger TABLE_OR_VIEW_NOT
    # _FOUND when the downstream action fires.
    #
    # The view name is derived deterministically from (tvf_sql,
    # id(df)) so reruns of the same cell reuse the same name and
    # createOrReplaceTempView keeps the session catalog bounded at
    # one entry per (cell × upstream) instead of growing one entry
    # per run. id(df) is included to keep two cells that happen to
    # have identical tvf_sql but distinct upstream DataFrames from
    # clobbering each other's bindings.
    tvf_sql: str = config.get("tvf_sql") or ""
    if tvf_sql:
        key = f"{tvf_sql}\x00{id(df)}".encode("utf-8")
        view_name = f"lakebuilder_ai_fn_{hashlib.sha256(key).hexdigest()[:12]}"
        df.createOrReplaceTempView(view_name)
        sql = tvf_sql.replace("__lakebuilder_ai_function_input__", view_name)
        return {"ai_data": spark.sql(sql)}

    expressions: List[str] = config.get("expressions", [])
    if not expressions:
        return {"ai_data": df}

    keep_all_columns: bool = config.get("keep_all_columns", True)
    if keep_all_columns:
        return {"ai_data": df.selectExpr(*expressions, "*")}
    return {"ai_data": df.selectExpr(*expressions)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "ai_classify(comment, '[\"santé\",\"famille\",\"formation\",\"lié au travail\",\"personnel\"]') `reason_category_json`"
    ],
    "keep_all_columns": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["unique_o5p6q7r8.unique_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["ai_function_20.ai_data"] = out["ai_data"]
if globals().get("ld_display_outputs", False) or "ai_function_20" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["ai_function_20.ai_data"])
if "ai_function_20" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "ai_function_20", "counts": {"ai_data": int(ctx["ai_function_20.ai_data"].count())}}}, raw=True)

In [0]:
"""
id: output_c2d3e4f5
template: output
templateVersion: 4.0.0
name: gold_absenteeism_by_department
parentId: group_b2c3d4e5
position:
  x: 1230
  y: 155.5
description:
  text: Create or refresh a materialized view with automatic clustering, based on input data.
  hash: fffe5e96
codegenHash: e556653ac6e0f4e0
config:
  output_type: materialized_view
  catalog: alp_demo_catalog
  schema: acme_hr
  view_name: gold_absenteeism_by_department
  cluster_by_mode: auto
input:
  - node: prepare_b5c6d7e8
    input_port: data
    output_port: prepared_data
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_FAST_CSV_CELLS = 50_000_000

MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES = 2 * 1024 * 1024 * 1024

_FAST_CSV_BYTES_PER_CELL_OVERHEAD = 16

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _pandas_line_terminator_parameter(pd) -> str:
    import inspect

    return (
        "lineterminator"
        if "lineterminator" in inspect.signature(pd.DataFrame.to_csv).parameters
        else "line_terminator"
    )

def _write_csv_overwrite(df, path: str, include_header: bool = True) -> bool:
    import os
    import shutil
    import tempfile

    from pyspark.sql import functions as F
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return False

    supported_type_names = {
        "boolean",
        "byte",
        "char",
        "date",
        "decimal",
        "double",
        "float",
        "integer",
        "long",
        "short",
        "string",
        "timestamp",
        "timestamp_ntz",
        "varchar",
        "void",
    }
    normalized_names = [name.casefold() for name in df.columns]
    if (
        not df.columns
        or len(set(normalized_names)) != len(normalized_names)
        or any(field.dataType.typeName() not in supported_type_names for field in df.schema.fields)
    ):
        return False

    spark_output_directory = os.path.isdir(path) and os.path.isfile(
        os.path.join(path, "_SUCCESS")
    )
    if os.path.isdir(path) and not spark_output_directory:
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )

    from databricks.sdk import WorkspaceClient

    client = WorkspaceClient()

    expressions = []
    output_column_names = []
    nan_markers = []
    for index, field in enumerate(df.schema.fields):
        escaped_name = field.name.replace("`", "``")
        column = F.col(f"`{escaped_name}`")
        output_column_name = f"__lb_csv_column_{index}"
        output_column_names.append(field.name)
        type_name = field.dataType.typeName()
        if type_name in {"char", "string", "varchar"}:
            first_character = F.substring(column, 1, 1)
            column = F.when(
                first_character.isin(*_FORMULA_TRIGGERS),
                F.concat(F.lit("'"), column),
            ).otherwise(column)
        elif type_name in {"byte", "integer", "long", "short"}:
            column = column.cast("string")
        elif type_name in {"double", "float"}:
            marker = f"__lb_csv_nan_{index}"
            nan_markers.append((output_column_name, marker))
            expressions.append(F.isnan(column).alias(marker))
        elif type_name == "boolean":
            column = F.when(column, F.lit("True")).when(~column, F.lit("False"))
        elif type_name in {"date", "decimal"}:
            column = column.cast("string")
        elif type_name in {"timestamp", "timestamp_ntz"}:
            column = F.regexp_replace(
                F.date_format(column, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
                r"[.]000000$",
                "",
            )
        expressions.append(column.alias(output_column_name))

    fast_csv_row_cap = min(
        MAX_SINGLE_FILE_ROWS,
        max(1, MAX_FAST_CSV_CELLS // len(df.columns)),
    )
    prepared_df = df.select(*expressions).limit(fast_csv_row_cap + 1).persist()
    try:
        payload_size = F.lit(0)
        for name in prepared_df.columns:
            escaped_name = name.replace("`", "``")
            payload_size = payload_size + F.length(
                F.encode(
                    F.coalesce(F.col(f"`{escaped_name}`").cast("string"), F.lit("")),
                    "UTF-8",
                )
            )
        stats = prepared_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.sum(payload_size).alias("payload_bytes"),
        ).collect()[0]
        row_count = stats["row_count"]
        if row_count > fast_csv_row_cap:
            if fast_csv_row_cap < MAX_SINGLE_FILE_ROWS:
                raise ValueError(
                    f"Output: result too wide for safe single-file CSV export "
                    f"(over {MAX_FAST_CSV_CELLS:,} cells) — write to a table instead."
                )
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
        estimated_driver_bytes = (stats["payload_bytes"] or 0) + (
            row_count * len(prepared_df.columns) * _FAST_CSV_BYTES_PER_CELL_OVERHEAD
        )
        if estimated_driver_bytes > MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES:
            raise ValueError(
                f"Output: result too large for safe single-file CSV export "
                f"(estimated driver memory exceeds "
                f"{MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES // (1024 * 1024):,} MiB) — "
                f"write to a table instead."
            )
        pdf = prepared_df.toPandas()
    finally:
        prepared_df.unpersist()
    for column_name, marker in nan_markers:
        nan_mask = pdf[marker].fillna(False)
        finite_mask = ~pdf[column_name].isna()
        pdf[column_name] = pdf[column_name].astype(object)
        pdf.loc[finite_mask, column_name] = pdf.loc[finite_mask, column_name].map(
            lambda value: repr(float(value))
        )
        pdf.loc[nan_mask, column_name] = "nan"
        pdf.loc[~nan_mask & ~finite_mask, column_name] = None
        pdf.drop(columns=[marker], inplace=True)
    pdf.columns = output_column_names
    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        pdf.to_csv(
            local_tmp,
            index=False,
            header=include_header,
            **{_pandas_line_terminator_parameter(pd): "\r\n"},
        )
        if spark_output_directory:
            shutil.rmtree(path)
        with open(local_tmp, "rb") as f:
            client.files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)
    return True

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    if (
        file_type == "csv"
        and not split_by_column
        and write_mode == "overwrite"
        and config.get("__fast_csv_overwrite", False)
        and _write_csv_overwrite(df, path, include_header)
    ):
        return

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "materialized_view",
    "catalog": "alp_demo_catalog",
    "schema": "acme_hr",
    "view_name": "gold_absenteeism_by_department",
    "cluster_by_mode": "auto"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_b5c6d7e8.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    try:
        _lb_disabled_persisted_result = spark.sql("SELECT * FROM alp_demo_catalog.acme_hr.gold_absenteeism_by_department").limit(0)
        _lb_disabled_persisted_result.schema
    except Exception as _lb_disabled_persisted_error:
        _lb_disabled_persisted_condition_getter = getattr(_lb_disabled_persisted_error, "getCondition", None)
        _lb_disabled_persisted_error_class = _lb_disabled_persisted_condition_getter() if callable(_lb_disabled_persisted_condition_getter) else None
        if not _lb_disabled_persisted_error_class:
            _lb_disabled_persisted_legacy_error_class_getter = getattr(_lb_disabled_persisted_error, "getErrorClass", None)
            _lb_disabled_persisted_error_class = _lb_disabled_persisted_legacy_error_class_getter() if callable(_lb_disabled_persisted_legacy_error_class_getter) else None
        if _lb_disabled_persisted_error_class not in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND"):
            raise
        _lb_disabled_persisted_input = inputs.get("data")
        _lb_disabled_persisted_result = _lb_disabled_persisted_input.limit(0) if hasattr(_lb_disabled_persisted_input, "limit") else _lb_disabled_persisted_input
    out = {"result": _lb_disabled_persisted_result}
else:
    out = run(config, inputs, spark)
ctx["output_c2d3e4f5.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_c2d3e4f5" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_c2d3e4f5.result"])
if "output_c2d3e4f5" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_c2d3e4f5", "counts": {"result": int(ctx["output_c2d3e4f5.result"].count())}}}, raw=True)

In [0]:
"""
id: prepare_q1r2s3t4
template: prepare
templateVersion: 1.0.0
name: Extract reason_category
parentId: group_c3d4e5f6
position:
  x: 1230
  y: 121.75
description:
  text: Create a new column 'reason_category' by extracting the first response from 'reason_category_json'.
  hash: c75576d6
codegenHash: 0517a7ccfa379cb8
config:
  actions:
    - type: formula
      target: reason_category
      expression: reason_category_json:response[0]::STRING
input:
  - node: ai_function_20
    input_port: data
    output_port: ai_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "reason_category",
            "expression": "reason_category_json:response[0]::STRING"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["ai_function_20.ai_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_q1r2s3t4.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_q1r2s3t4" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_q1r2s3t4.prepared_data"])
if "prepare_q1r2s3t4" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "prepare_q1r2s3t4", "counts": {"prepared_data": int(ctx["prepare_q1r2s3t4.prepared_data"].count())}}}, raw=True)

In [0]:
"""
id: join_w3x4y5z6
template: join
templateVersion: 3.0.0
name: Absences with reasons
parentId: group_c3d4e5f6
position:
  x: 1530
  y: 78
description:
  text: Combine two data sets based on matching comment values, keeping only those with matching comments from both sides.
  hash: 485c7247
previewCodeHash: d92852e58adf2a3f
codegenHash: ea5821d113ebbb3b
config:
  join_type: inner
  match_case: true
  multi_table: false
  conditions:
    - left_input: 0
      right_input: 1
      join_keys:
        - left: comment
          right: comment
      join_conditions: ""
  columns:
    - edits: []
      ordered: []
    - edits:
        - column: reason_category_json
          checked: false
        - column: absence_id
          checked: false
        - column: employee_id
          checked: false
        - column: absence_type
          checked: false
        - column: start_date
          checked: false
        - column: end_date
          checked: false
        - column: days
          checked: false
        - column: event_timestamp
          checked: false
        - column: absence_month
          checked: false
      ordered: []
input:
  - node: source_g7h8i9j0
    input_port: left
    output_port: data
  - node: prepare_q1r2s3t4
    input_port: right
    output_port: prepared_data
"""

# generated from the system
from typing import Any, Dict, List, Optional
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(alias: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + alias + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(frames: List[Any]):
    out = []
    seen: Dict[str, int] = {}
    n = len(frames)
    for idx, (alias, df, cfg) in enumerate(frames):
        edits = _edits_by_column(cfg)
        for name in _ordered_names(list(df.columns), cfg):
            edit = edits.get(name)
            if edit is not None and not _is_checked(edit):
                continue
            col = _qualified_col(alias, name)
            user_alias = edit.get("alias") if edit else None
            if user_alias:
                out.append(col.alias(user_alias))
                seen[user_alias.lower()] = seen.get(user_alias.lower(), 0) + 1
                continue
            key = name.lower()
            if key not in seen:
                seen[key] = 1
                out.append(col.alias(name))
                continue
            new_name = ("right_" + name) if (n == 2 and idx == 1) else (name + "_" + str(idx + 1))
            while new_name.lower() in seen:
                new_name = new_name + "_dup"
            seen[new_name.lower()] = 1
            out.append(col.alias(new_name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = []
    for port in ("left", "right"):
        df = inputs.get(port)
        if df is not None:
            dfs.append(df)
    dfs.extend(inputs.get("data") or [])
    if len(dfs) < 2:
        raise ValueError(
            f"Join requires at least 2 inputs, got {len(dfs)}. "
            "Wire 2 or more upstream operators into this Join."
        )

    n = len(dfs)
    join_type = config.get("join_type") or "split_join"
    match_case = bool(config.get("match_case", False))
    conditions: List[Dict[str, Any]] = config.get("conditions") or []
    columns: List[Dict[str, Any]] = config.get("columns") or []

    aliased = [df.alias("t" + str(i)) for i, df in enumerate(dfs)]
    types = [{f.name.lower(): f.dataType for f in df.schema} for df in aliased]

    def key_col(idx: int, name: str):
        col = _qualified_col("t" + str(idx), name)
        if not match_case and isinstance(types[idx].get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    def edge_predicate(i: int, j: int, edge: Dict[str, Any]) -> Optional[Any]:
        predicates = []
        for key in edge.get("join_keys") or []:
            predicates.append(key_col(i, key["left"]) == key_col(j, key["right"]))
        condition = edge.get("join_conditions") or ""
        if condition:
            predicates.append(F.expr(condition))
        join_expr = None
        for predicate in predicates:
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    edges: List[Any] = [
        (int(edge.get("left_input", 0)), int(edge.get("right_input", 0)), edge)
        for edge in conditions
    ]

    predicates_by_step: Dict[int, List[Any]] = {}
    for i, j, edge in edges:
        predicate = edge_predicate(i, j, edge)
        if predicate is not None:
            predicates_by_step.setdefault(max(i, j), []).append(predicate)

    def predicate_at(step: int) -> Optional[Any]:
        join_expr = None
        for predicate in predicates_by_step.get(step, []):
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    def cfg(i: int) -> Dict[str, Any]:
        return columns[i] if i < len(columns) else {}

    is_two = n == 2
    is_split = is_two and join_type == "split_join"

    if is_split:
        join_expr = predicate_at(1)
        left_df, right_df = aliased[0], aliased[1]
        if join_expr is None:
            matched = left_df.join(right_df, how="inner")
            left_unmatched = left_df.join(right_df, how="left_anti")
            right_unmatched = right_df.join(left_df, how="left_anti")
        else:
            matched = left_df.join(right_df, join_expr, how="inner")
            left_unmatched = left_df.join(right_df, join_expr, how="left_anti")
            right_unmatched = right_df.join(left_df, join_expr, how="left_anti")
        projection = _projection([("t0", left_df, cfg(0)), ("t1", right_df, cfg(1))])
        if projection:
            matched = matched.select(*projection)
        return {
            "joined_data": matched,
            "left_unmatched": left_unmatched,
            "right_unmatched": right_unmatched,
        }

    how = join_type if is_two else "inner"
    if is_two:
        acc = aliased[0]
        predicate = predicate_at(1)
        if predicate is None:
            acc = acc.join(aliased[1], how=how)
        else:
            acc = acc.join(aliased[1], predicate, how=how)
    else:
        adjacency: Dict[int, List[int]] = {}
        for idx, (i, j, _edge) in enumerate(edges):
            adjacency.setdefault(i, []).append(idx)
            adjacency.setdefault(j, []).append(idx)
        visited = [False] * n
        visited[0] = True
        acc = aliased[0]
        used_edges = set()
        queue: List[int] = [0]
        while queue:
            cur = queue.pop(0)
            for idx in adjacency.get(cur, []):
                i, j, edge = edges[idx]
                nxt = j if i == cur else i
                if visited[nxt]:
                    continue
                visited[nxt] = True
                used_edges.add(idx)
                predicate = edge_predicate(i, j, edge)
                if predicate is None:
                    acc = acc.join(aliased[nxt], how=how)
                else:
                    acc = acc.join(aliased[nxt], predicate, how=how)
                queue.append(nxt)
        for idx, (i, j, edge) in enumerate(edges):
            if idx in used_edges:
                continue
            predicate = edge_predicate(i, j, edge)
            if predicate is not None:
                acc = acc.where(predicate)

    projection = _projection([("t" + str(i), aliased[i], cfg(i)) for i in range(n)])
    if projection:
        acc = acc.select(*projection)

    return {
        "joined_data": acc,
        "left_unmatched": spark.createDataFrame([], aliased[0].schema),
        "right_unmatched": spark.createDataFrame([], aliased[n - 1].schema),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "inner",
    "match_case": True,
    "multi_table": False,
    "conditions": [
        {
            "left_input": 0,
            "right_input": 1,
            "join_keys": [
                {
                    "left": "comment",
                    "right": "comment"
                }
            ],
            "join_conditions": ""
        }
    ],
    "columns": [
        {
            "edits": [],
            "ordered": []
        },
        {
            "edits": [
                {
                    "column": "reason_category_json",
                    "checked": False
                },
                {
                    "column": "absence_id",
                    "checked": False
                },
                {
                    "column": "employee_id",
                    "checked": False
                },
                {
                    "column": "absence_type",
                    "checked": False
                },
                {
                    "column": "start_date",
                    "checked": False
                },
                {
                    "column": "end_date",
                    "checked": False
                },
                {
                    "column": "days",
                    "checked": False
                },
                {
                    "column": "event_timestamp",
                    "checked": False
                },
                {
                    "column": "absence_month",
                    "checked": False
                }
            ],
            "ordered": []
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "left": ctx["source_g7h8i9j0.data"],
    "right": ctx["prepare_q1r2s3t4.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["join_w3x4y5z6.joined_data"] = out["joined_data"]
ctx["join_w3x4y5z6.left_unmatched"] = out["left_unmatched"]
ctx["join_w3x4y5z6.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", False) or "join_w3x4y5z6" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["join_w3x4y5z6.joined_data"])
    display(ctx["join_w3x4y5z6.left_unmatched"])
    display(ctx["join_w3x4y5z6.right_unmatched"])
if "join_w3x4y5z6" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "join_w3x4y5z6", "counts": {"joined_data": int(ctx["join_w3x4y5z6.joined_data"].count()), "left_unmatched": int(ctx["join_w3x4y5z6.left_unmatched"].count()), "right_unmatched": int(ctx["join_w3x4y5z6.right_unmatched"].count())}}}, raw=True)

In [0]:
"""
id: join_e6f7g8h9
template: join
templateVersion: 3.0.0
name: Reasons with employees
parentId: group_c3d4e5f6
position:
  x: 1830
  y: 155.5
previewCodeHash: 6e76bd455a813e2b
codegenHash: ea5821d113ebbb3b
config:
  join_type: inner
  match_case: false
  multi_table: false
  conditions:
    - left_input: 0
      right_input: 1
      join_keys:
        - left: employee_id
          right: employee_id
      join_conditions: ""
  columns:
    - edits: []
      ordered: []
    - edits: []
      ordered: []
input:
  - node: join_w3x4y5z6
    input_port: left
    output_port: joined_data
  - node: source_a1b2c3d5
    input_port: right
    output_port: data
"""

# generated from the system
from typing import Any, Dict, List, Optional
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(alias: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + alias + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(frames: List[Any]):
    out = []
    seen: Dict[str, int] = {}
    n = len(frames)
    for idx, (alias, df, cfg) in enumerate(frames):
        edits = _edits_by_column(cfg)
        for name in _ordered_names(list(df.columns), cfg):
            edit = edits.get(name)
            if edit is not None and not _is_checked(edit):
                continue
            col = _qualified_col(alias, name)
            user_alias = edit.get("alias") if edit else None
            if user_alias:
                out.append(col.alias(user_alias))
                seen[user_alias.lower()] = seen.get(user_alias.lower(), 0) + 1
                continue
            key = name.lower()
            if key not in seen:
                seen[key] = 1
                out.append(col.alias(name))
                continue
            new_name = ("right_" + name) if (n == 2 and idx == 1) else (name + "_" + str(idx + 1))
            while new_name.lower() in seen:
                new_name = new_name + "_dup"
            seen[new_name.lower()] = 1
            out.append(col.alias(new_name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = []
    for port in ("left", "right"):
        df = inputs.get(port)
        if df is not None:
            dfs.append(df)
    dfs.extend(inputs.get("data") or [])
    if len(dfs) < 2:
        raise ValueError(
            f"Join requires at least 2 inputs, got {len(dfs)}. "
            "Wire 2 or more upstream operators into this Join."
        )

    n = len(dfs)
    join_type = config.get("join_type") or "split_join"
    match_case = bool(config.get("match_case", False))
    conditions: List[Dict[str, Any]] = config.get("conditions") or []
    columns: List[Dict[str, Any]] = config.get("columns") or []

    aliased = [df.alias("t" + str(i)) for i, df in enumerate(dfs)]
    types = [{f.name.lower(): f.dataType for f in df.schema} for df in aliased]

    def key_col(idx: int, name: str):
        col = _qualified_col("t" + str(idx), name)
        if not match_case and isinstance(types[idx].get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    def edge_predicate(i: int, j: int, edge: Dict[str, Any]) -> Optional[Any]:
        predicates = []
        for key in edge.get("join_keys") or []:
            predicates.append(key_col(i, key["left"]) == key_col(j, key["right"]))
        condition = edge.get("join_conditions") or ""
        if condition:
            predicates.append(F.expr(condition))
        join_expr = None
        for predicate in predicates:
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    edges: List[Any] = [
        (int(edge.get("left_input", 0)), int(edge.get("right_input", 0)), edge)
        for edge in conditions
    ]

    predicates_by_step: Dict[int, List[Any]] = {}
    for i, j, edge in edges:
        predicate = edge_predicate(i, j, edge)
        if predicate is not None:
            predicates_by_step.setdefault(max(i, j), []).append(predicate)

    def predicate_at(step: int) -> Optional[Any]:
        join_expr = None
        for predicate in predicates_by_step.get(step, []):
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    def cfg(i: int) -> Dict[str, Any]:
        return columns[i] if i < len(columns) else {}

    is_two = n == 2
    is_split = is_two and join_type == "split_join"

    if is_split:
        join_expr = predicate_at(1)
        left_df, right_df = aliased[0], aliased[1]
        if join_expr is None:
            matched = left_df.join(right_df, how="inner")
            left_unmatched = left_df.join(right_df, how="left_anti")
            right_unmatched = right_df.join(left_df, how="left_anti")
        else:
            matched = left_df.join(right_df, join_expr, how="inner")
            left_unmatched = left_df.join(right_df, join_expr, how="left_anti")
            right_unmatched = right_df.join(left_df, join_expr, how="left_anti")
        projection = _projection([("t0", left_df, cfg(0)), ("t1", right_df, cfg(1))])
        if projection:
            matched = matched.select(*projection)
        return {
            "joined_data": matched,
            "left_unmatched": left_unmatched,
            "right_unmatched": right_unmatched,
        }

    how = join_type if is_two else "inner"
    if is_two:
        acc = aliased[0]
        predicate = predicate_at(1)
        if predicate is None:
            acc = acc.join(aliased[1], how=how)
        else:
            acc = acc.join(aliased[1], predicate, how=how)
    else:
        adjacency: Dict[int, List[int]] = {}
        for idx, (i, j, _edge) in enumerate(edges):
            adjacency.setdefault(i, []).append(idx)
            adjacency.setdefault(j, []).append(idx)
        visited = [False] * n
        visited[0] = True
        acc = aliased[0]
        used_edges = set()
        queue: List[int] = [0]
        while queue:
            cur = queue.pop(0)
            for idx in adjacency.get(cur, []):
                i, j, edge = edges[idx]
                nxt = j if i == cur else i
                if visited[nxt]:
                    continue
                visited[nxt] = True
                used_edges.add(idx)
                predicate = edge_predicate(i, j, edge)
                if predicate is None:
                    acc = acc.join(aliased[nxt], how=how)
                else:
                    acc = acc.join(aliased[nxt], predicate, how=how)
                queue.append(nxt)
        for idx, (i, j, edge) in enumerate(edges):
            if idx in used_edges:
                continue
            predicate = edge_predicate(i, j, edge)
            if predicate is not None:
                acc = acc.where(predicate)

    projection = _projection([("t" + str(i), aliased[i], cfg(i)) for i in range(n)])
    if projection:
        acc = acc.select(*projection)

    return {
        "joined_data": acc,
        "left_unmatched": spark.createDataFrame([], aliased[0].schema),
        "right_unmatched": spark.createDataFrame([], aliased[n - 1].schema),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "inner",
    "match_case": False,
    "multi_table": False,
    "conditions": [
        {
            "left_input": 0,
            "right_input": 1,
            "join_keys": [
                {
                    "left": "employee_id",
                    "right": "employee_id"
                }
            ],
            "join_conditions": ""
        }
    ],
    "columns": [
        {
            "edits": [],
            "ordered": []
        },
        {
            "edits": [],
            "ordered": []
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "left": ctx["join_w3x4y5z6.joined_data"],
    "right": ctx["source_a1b2c3d5.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["join_e6f7g8h9.joined_data"] = out["joined_data"]
ctx["join_e6f7g8h9.left_unmatched"] = out["left_unmatched"]
ctx["join_e6f7g8h9.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", False) or "join_e6f7g8h9" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["join_e6f7g8h9.joined_data"])
    display(ctx["join_e6f7g8h9.left_unmatched"])
    display(ctx["join_e6f7g8h9.right_unmatched"])
if "join_e6f7g8h9" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "join_e6f7g8h9", "counts": {"joined_data": int(ctx["join_e6f7g8h9.joined_data"].count()), "left_unmatched": int(ctx["join_e6f7g8h9.left_unmatched"].count()), "right_unmatched": int(ctx["join_e6f7g8h9.right_unmatched"].count())}}}, raw=True)

In [0]:
"""
id: sql_i0j1k2l3
template: aggregate
templateVersion: 2.0.0
name: Absence reason metrics
parentId: group_c3d4e5f6
position:
  x: 2130
  y: 155.5
description:
  text: Group by department name, business unit, and reason category. Count absence events and sum total absence days.
  hash: f16470df
previewCodeHash: bab69477b1410b45
codegenHash: 7e9149ee54d999b7
config:
  group_bys:
    - expr: department_name
      type: expr
    - expr: business_unit
      type: expr
    - expr: reason_category
      type: expr
  aggregations:
    - columnExpr:
        expr: absence_id
        type: expr
      fn: COUNT
      alias: absence_events
    - columnExpr:
        expr: days
        type: expr
      fn: SUM
      alias: total_absence_days
input:
  - node: join_e6f7g8h9
    input_port: data
    output_port: joined_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = agg_fn(arg)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            concat_arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = F.concat_ws(sep, F.collect_list(concat_arg))
        elif fn == "COUNT_DISTINCT":
            arg = F.expr(raw_expr) if col_expr.get("type") != "column" else raw_expr
            col = F.count_distinct(arg)
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "department_name",
            "type": "expr"
        },
        {
            "expr": "business_unit",
            "type": "expr"
        },
        {
            "expr": "reason_category",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "absence_id",
                "type": "expr"
            },
            "fn": "COUNT",
            "alias": "absence_events",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "days",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "total_absence_days",
            "withAsKeyword": None
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["join_e6f7g8h9.joined_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["sql_i0j1k2l3.aggregated_data"] = out["aggregated_data"]
if globals().get("ld_display_outputs", False) or "sql_i0j1k2l3" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["sql_i0j1k2l3.aggregated_data"])
if "sql_i0j1k2l3" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "sql_i0j1k2l3", "counts": {"aggregated_data": int(ctx["sql_i0j1k2l3.aggregated_data"].count())}}}, raw=True)

In [0]:
"""
id: output_m4n5o6p7
template: output
templateVersion: 4.0.0
name: gold_absence_reasons_by_department
parentId: group_c3d4e5f6
position:
  x: 2430
  y: 155.5
codegenHash: e556653ac6e0f4e0
config:
  output_type: materialized_view
  catalog: alp_demo_catalog
  schema: acme_hr
  view_name: gold_absence_reasons_by_department
  cluster_by_mode: auto
input:
  - node: sql_i0j1k2l3
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_FAST_CSV_CELLS = 50_000_000

MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES = 2 * 1024 * 1024 * 1024

_FAST_CSV_BYTES_PER_CELL_OVERHEAD = 16

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _pandas_line_terminator_parameter(pd) -> str:
    import inspect

    return (
        "lineterminator"
        if "lineterminator" in inspect.signature(pd.DataFrame.to_csv).parameters
        else "line_terminator"
    )

def _write_csv_overwrite(df, path: str, include_header: bool = True) -> bool:
    import os
    import shutil
    import tempfile

    from pyspark.sql import functions as F
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return False

    supported_type_names = {
        "boolean",
        "byte",
        "char",
        "date",
        "decimal",
        "double",
        "float",
        "integer",
        "long",
        "short",
        "string",
        "timestamp",
        "timestamp_ntz",
        "varchar",
        "void",
    }
    normalized_names = [name.casefold() for name in df.columns]
    if (
        not df.columns
        or len(set(normalized_names)) != len(normalized_names)
        or any(field.dataType.typeName() not in supported_type_names for field in df.schema.fields)
    ):
        return False

    spark_output_directory = os.path.isdir(path) and os.path.isfile(
        os.path.join(path, "_SUCCESS")
    )
    if os.path.isdir(path) and not spark_output_directory:
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )

    from databricks.sdk import WorkspaceClient

    client = WorkspaceClient()

    expressions = []
    output_column_names = []
    nan_markers = []
    for index, field in enumerate(df.schema.fields):
        escaped_name = field.name.replace("`", "``")
        column = F.col(f"`{escaped_name}`")
        output_column_name = f"__lb_csv_column_{index}"
        output_column_names.append(field.name)
        type_name = field.dataType.typeName()
        if type_name in {"char", "string", "varchar"}:
            first_character = F.substring(column, 1, 1)
            column = F.when(
                first_character.isin(*_FORMULA_TRIGGERS),
                F.concat(F.lit("'"), column),
            ).otherwise(column)
        elif type_name in {"byte", "integer", "long", "short"}:
            column = column.cast("string")
        elif type_name in {"double", "float"}:
            marker = f"__lb_csv_nan_{index}"
            nan_markers.append((output_column_name, marker))
            expressions.append(F.isnan(column).alias(marker))
        elif type_name == "boolean":
            column = F.when(column, F.lit("True")).when(~column, F.lit("False"))
        elif type_name in {"date", "decimal"}:
            column = column.cast("string")
        elif type_name in {"timestamp", "timestamp_ntz"}:
            column = F.regexp_replace(
                F.date_format(column, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
                r"[.]000000$",
                "",
            )
        expressions.append(column.alias(output_column_name))

    fast_csv_row_cap = min(
        MAX_SINGLE_FILE_ROWS,
        max(1, MAX_FAST_CSV_CELLS // len(df.columns)),
    )
    prepared_df = df.select(*expressions).limit(fast_csv_row_cap + 1).persist()
    try:
        payload_size = F.lit(0)
        for name in prepared_df.columns:
            escaped_name = name.replace("`", "``")
            payload_size = payload_size + F.length(
                F.encode(
                    F.coalesce(F.col(f"`{escaped_name}`").cast("string"), F.lit("")),
                    "UTF-8",
                )
            )
        stats = prepared_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.sum(payload_size).alias("payload_bytes"),
        ).collect()[0]
        row_count = stats["row_count"]
        if row_count > fast_csv_row_cap:
            if fast_csv_row_cap < MAX_SINGLE_FILE_ROWS:
                raise ValueError(
                    f"Output: result too wide for safe single-file CSV export "
                    f"(over {MAX_FAST_CSV_CELLS:,} cells) — write to a table instead."
                )
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
        estimated_driver_bytes = (stats["payload_bytes"] or 0) + (
            row_count * len(prepared_df.columns) * _FAST_CSV_BYTES_PER_CELL_OVERHEAD
        )
        if estimated_driver_bytes > MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES:
            raise ValueError(
                f"Output: result too large for safe single-file CSV export "
                f"(estimated driver memory exceeds "
                f"{MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES // (1024 * 1024):,} MiB) — "
                f"write to a table instead."
            )
        pdf = prepared_df.toPandas()
    finally:
        prepared_df.unpersist()
    for column_name, marker in nan_markers:
        nan_mask = pdf[marker].fillna(False)
        finite_mask = ~pdf[column_name].isna()
        pdf[column_name] = pdf[column_name].astype(object)
        pdf.loc[finite_mask, column_name] = pdf.loc[finite_mask, column_name].map(
            lambda value: repr(float(value))
        )
        pdf.loc[nan_mask, column_name] = "nan"
        pdf.loc[~nan_mask & ~finite_mask, column_name] = None
        pdf.drop(columns=[marker], inplace=True)
    pdf.columns = output_column_names
    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        pdf.to_csv(
            local_tmp,
            index=False,
            header=include_header,
            **{_pandas_line_terminator_parameter(pd): "\r\n"},
        )
        if spark_output_directory:
            shutil.rmtree(path)
        with open(local_tmp, "rb") as f:
            client.files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)
    return True

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    if (
        file_type == "csv"
        and not split_by_column
        and write_mode == "overwrite"
        and config.get("__fast_csv_overwrite", False)
        and _write_csv_overwrite(df, path, include_header)
    ):
        return

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "materialized_view",
    "catalog": "alp_demo_catalog",
    "schema": "acme_hr",
    "view_name": "gold_absence_reasons_by_department",
    "cluster_by_mode": "auto"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["sql_i0j1k2l3.aggregated_data"]
}
if config["meta_state"]["is_disabled"]:
    try:
        _lb_disabled_persisted_result = spark.sql("SELECT * FROM alp_demo_catalog.acme_hr.gold_absence_reasons_by_department").limit(0)
        _lb_disabled_persisted_result.schema
    except Exception as _lb_disabled_persisted_error:
        _lb_disabled_persisted_condition_getter = getattr(_lb_disabled_persisted_error, "getCondition", None)
        _lb_disabled_persisted_error_class = _lb_disabled_persisted_condition_getter() if callable(_lb_disabled_persisted_condition_getter) else None
        if not _lb_disabled_persisted_error_class:
            _lb_disabled_persisted_legacy_error_class_getter = getattr(_lb_disabled_persisted_error, "getErrorClass", None)
            _lb_disabled_persisted_error_class = _lb_disabled_persisted_legacy_error_class_getter() if callable(_lb_disabled_persisted_legacy_error_class_getter) else None
        if _lb_disabled_persisted_error_class not in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND"):
            raise
        _lb_disabled_persisted_input = inputs.get("data")
        _lb_disabled_persisted_result = _lb_disabled_persisted_input.limit(0) if hasattr(_lb_disabled_persisted_input, "limit") else _lb_disabled_persisted_input
    out = {"result": _lb_disabled_persisted_result}
else:
    out = run(config, inputs, spark)
ctx["output_m4n5o6p7.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_m4n5o6p7" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_m4n5o6p7.result"])
if "output_m4n5o6p7" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_m4n5o6p7", "counts": {"result": int(ctx["output_m4n5o6p7.result"].count())}}}, raw=True)

---
id: group_a1b2c3d4
template: group
name: A – Headcount by department
position:
  x: 750
  y: 0
dimensions:
  width: 1160
  height: 203
---

---
id: group_b2c3d4e5
template: group
name: B – Absenteeism by department
position:
  x: 600
  y: 263
dimensions:
  width: 1460
  height: 358
---

---
id: group_c3d4e5f6
template: group
name: C – Absence reasons by department
position:
  x: 0
  y: 681
dimensions:
  width: 2660
  height: 358
---